In [ ]:
import os
import subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/Video-Depth-Anything")
ENCODER = "vits"
HF_REPOS = {
    "vits": "depth-anything/Video-Depth-Anything-Small",
    "vitb": "depth-anything/Video-Depth-Anything-Base",
    "vitl": "depth-anything/Video-Depth-Anything-Large",
}

if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/DepthAnything/Video-Depth-Anything.git", str(CODE_DIR)], check=True)

os.chdir(CODE_DIR)

runtime_deps = [
    "opencv-python",
    "matplotlib",
    "pillow",
    "imageio==2.37.0",
    "imageio-ffmpeg==0.4.7",
    "einops==0.4.1",
    "easydict",
    "tqdm",
    "huggingface_hub",
]
subprocess.run(["pip", "-q", "install", *runtime_deps], check=True)
subprocess.run(["pip", "-q", "install", "decord"], check=False)

run_py = CODE_DIR / "run.py"
content = run_py.read_text()
old = "torch.load(f'./checkpoints/{checkpoint_name}_{args.encoder}.pth', map_location='cpu')"
new = "torch.load(f'./checkpoints/{checkpoint_name}_{args.encoder}.pth', map_location='cpu', weights_only=False)"
if old in content and new not in content:
    run_py.write_text(content.replace(old, new))
    print("Patched run.py for weights_only=False")

checkpoint_dir = CODE_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / f"video_depth_anything_{ENCODER}.pth"

if checkpoint_path.exists() and checkpoint_path.stat().st_size < 100_000_000:
    print("Removing invalid checkpoint:", checkpoint_path, checkpoint_path.stat().st_size, "bytes")
    checkpoint_path.unlink()

if not checkpoint_path.exists():
    from huggingface_hub import hf_hub_download

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        secret_value_0 = user_secrets.get_secret("hf-t1")
        HF_TOKEN = secret_value_0
        print("Using hf-t1 from Kaggle secrets")
    except Exception:
        HF_TOKEN = None
        print("hf-t1 not found; downloading anonymously")

    downloaded = Path(hf_hub_download(
        repo_id=HF_REPOS[ENCODER],
        filename=f"video_depth_anything_{ENCODER}.pth",
        local_dir=str(checkpoint_dir),
        token=HF_TOKEN,
    ))
    if downloaded != checkpoint_path:
        downloaded.replace(checkpoint_path)

if checkpoint_path.stat().st_size < 100_000_000:
    preview = checkpoint_path.read_text(errors="ignore")[:200]
    raise RuntimeError(f"Checkpoint download failed: {checkpoint_path} is only {checkpoint_path.stat().st_size} bytes. Preview: {preview!r}")

subprocess.run(["ls", "-lh", str(checkpoint_path)], check=True)

In [ ]:
import os
import glob
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = "/kaggle/input/datasets/dharun235/lateral-insect-view/Lateral view insect video"
OUTPUT_ROOT = "/kaggle/working/MDE_Outputs"
CODE_DIR = "/kaggle/working/Video-Depth-Anything"

VIDEO_EXTENSIONS = ("*.mp4", "*.avi", "*.mov", "*.mkv")
ENCODER = "vits"
SKIP_EXISTING = True

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.chdir(CODE_DIR)


def output_stem_for(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    parent = os.path.relpath(os.path.dirname(video_path), INPUT_ROOT)
    if parent in (".", ""):
        return video_stem
    return f"{parent.replace(os.sep, '_')}_{video_stem}"


video_list = sorted(
    video_path
    for pattern in VIDEO_EXTENSIONS
    for video_path in glob.glob(os.path.join(INPUT_ROOT, "**", pattern), recursive=True)
)

print(f"Found {len(video_list)} videos.")

for idx, video_path in enumerate(video_list, 1):
    output_stem = output_stem_for(video_path)
    output_video = os.path.join(OUTPUT_ROOT, f"{output_stem}_vis.mp4")

    if SKIP_EXISTING and os.path.exists(output_video):
        print(f"[{idx}/{len(video_list)}] Skipping existing: {output_video}")
        continue

    print("=" * 80)
    print(f"[{idx}/{len(video_list)}] Processing: {video_path}")

    cmd = [
        sys.executable,
        "run.py",
        "--input_video",
        video_path,
        "--output_dir",
        OUTPUT_ROOT,
        "--encoder",
        ENCODER,
        "--input_size",
        "266",
        "--max_len",
        "1024",
        "--max_res",
        "512",
    ]

    try:
        subprocess.run(cmd, check=True)
        generated = os.path.join(OUTPUT_ROOT, f"{os.path.splitext(os.path.basename(video_path))[0]}_vis.mp4")
        if generated != output_video and os.path.exists(generated):
            os.replace(generated, output_video)
        print("Saved:", output_video)
    except subprocess.CalledProcessError as exc:
        print(f"Failed: {video_path}: {exc}")

print("Done. Outputs:", OUTPUT_ROOT)